# Subagent with an MCP Tool — A Subagent Is Just an LLMAgent

This notebook builds `websearch_subagent`, wired to an MCP-discovered
websearch tool, and registers it alongside the framework's `general`
and `explore` default recipes. Nothing about `SubAgentSpec` restricts
what a dispatched agent can be equipped with: `builder` is an ordinary
`LLMAgentBuilder`, the same one used for a top-level agent.

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have
Ollama installed on your local machine and have its LLM hosting
service running. To download Ollama, follow the instructions found on
this page: https://ollama.com/download. After downloading and
installing Ollama, you can start a service by opening a terminal and
running `ollama serve`.

In [2]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [3]:
model = "glm-5.3:cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

## Connecting to a Websearch MCP Server

[`duckduckgo-mcp-server`](https://github.com/nickclyde/duckduckgo-mcp-server)
exposes `search` and `fetch_content` tools over stdio. Unlike the
GitHub and GoodNews MCP servers from Chapter 5, it needs no API key
and no local clone; `uvx` fetches and runs it on demand, requiring
only that `uv` is installed on your machine.

In [4]:
from mcp import StdioServerParameters

from llm_agents_from_scratch.tools.mcp import MCPToolProvider

websearch_mcp_provider = MCPToolProvider(
    name="websearch_mcp",
    stdio_params=StdioServerParameters(
        command="uvx",
        args=["duckduckgo-mcp-server"],
    ),
)

## Defining the Roster

`websearch_subagent`'s builder is nothing special: an
`LLMAgentBuilder` pointed at the MCP provider above, the same builder
class every other subagent in this book uses. Registering it alongside
`general_subagent` and `explore_subagent` shows the coordinator's
subagent registry is an open list, not a fixed pair of defaults.

In [5]:
from llm_agents_from_scratch import LLMAgent, LLMAgentBuilder
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.subagents import SubAgentSpec
from llm_agents_from_scratch.subagents.recipes import (
    explore_subagent,
    general_subagent,
)

llm = OllamaLLM(
    host=host,
    model=model,
    think=use_cloud,
    json_prompt_mode=use_cloud,
)

websearch_subagent = SubAgentSpec(
    name="websearch_subagent",
    description=(
        "Searches the live web and fetches page content for current info."
    ),
    builder=LLMAgentBuilder(llm=llm, mcp_providers=[websearch_mcp_provider]),
    max_steps=5,
)

coordinator = LLMAgent(
    llm=llm,
    subagents=[
        general_subagent(llm),
        explore_subagent(llm),
        websearch_subagent,
    ],
)

## Example — Routing to the Websearch Specialist

The task below needs information no local file or general-purpose
subagent has: today's actual, current release of Python. Only
`websearch_subagent` can look that up.

In [6]:
task = Task(
    instruction=(
        "Ask websearch_subagent to search the web for the current "
        "official version number of the Python programming language "
        "and report it back with the source URL. Report exactly what "
        "it returns."
    ),
)
handler = coordinator.run(task, max_steps=10)
result = await handler

In [7]:
print(result.content)

The websearch_subagent returned the following result — reporting it exactly as received:

---

**Current official Python version: 3.14.7**

**Source URL:** https://www.python.org/downloads/latest/

**Verification details:**
- The official Python.org page is titled **"Python Release Python 3.14.7"**
- **Release date:** Aug. 5, 2026
- It is described as the **seventh maintenance release of Python 3.14**, containing around 499 bugfixes, build improvements, and documentation changes from 86 contributors since 3.14.6
- This was confirmed directly from the official python.org source, corroborating the earlier search results (which also referenced the official announcement on the Python Insider blog for Python 3.14.7 and 3.13.15)

Python 3.14 is the latest stable major release series (released October 7, 2025), and **3.14.7** is the current maintenance release within that series.

---

That's the subagent's report in full — the current official Python version is **3.14.7**, sourced from https

`handler.rollout` shows the coordinator dispatching straight to
`websearch_subagent`, the only registered specialist actually equipped
to answer; `general_subagent` and `explore_subagent` stay registered
but unused, exactly as a real open registry should behave.

In [8]:
print(handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask websearch_subagent to search the web for the current official version number of the Python programming language and report it back with the source URL. Report exactly what it returns.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "b38ec4d9-f762-4ca9-85c6-51c9a840dfa3",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "websearch_subagent",
        "task": "Search the web for the current official version number of the Python programming language. Report back the exact current official version number along with the source URL where you found this information."
    }
}.

🔧 tool: {
    "tool_call_id": "b38ec4d9-f762-4ca9-85c6-51c9a840dfa3",
    "content": "I've now directly verified the version from the official source. The Python.org downloads page confirms the version number.\n\n## Result\n\n**Current official Python version: 3.14.7**\n\n**Source URL:** https://w